# O‘zbekcha hikoyalar — LLM Midterm Dataset Pipeline

**Bitta notebookda barcha bosqichlar (Dataset Buildergacha).** Ushbu notebook alohida `src/*.py` fayllardagi funksiyalarni **ketma-ket ishlatadi**. Shuning uchun loyiha papkasi bilan birga ishlating.

**Maqsad:** yangi o‘zbekcha qisqa hikoyalardan LLM uchun tayyor `train.bin`, `val.bin`, `tokenizer.json` va statistika hisobotini yaratish.

> Dataset TinyStories emas. `data/source` ichidagi maxsus tuzilgan qisqa hikoyalar ishlatiladi. Litsenziya va cheklovlar: **DATASET_README.md**. Modelni o‘qitish bu notebookka kirmaydi.

**Bosqichlar:** 0. Manba topish va olish → 1. Cleaning → 2. Filtering → 3. Deduplication → 4. PII Scrubbing → 5. Statistika / Train-Val → 6. Tokenizer → 7. Dataset Builder → 8. Testlar.

## Ishga tushirish (eng boshida)

1. ZIP arxivni kompyuteringizga chiqaring.
2. Terminalni loyiha bosh papkasida oching: `pip install -r requirements.txt`.
3. VS Code yoki Jupyterda shu notebookni oching.
4. **Run All** ni bosing. Notebook loyiha joylashuvini o‘zi aniqlaydi.

**Clean clone sharti:** barcha zarur hikoyalar `data/source/hikoyalar_manba.jsonl` ichida. Pip paketlari o‘rnatilgan bo‘lsa, internet **shart emas**.

**Muhim:** kuzatish uchun yaratilgan `X...` misollar haqiqiy shaxsiy ma'lumotlarni emas, sinov yozuvlarini o‘z ichiga oladi.

In [1]:
# 0-a. Kutubxonalarni yuklaymiz va loyiha papkasini topamiz.
# Bu modul importlari Python fayllarini takror yozmaslikka yordam beradi.
from pathlib import Path
import sys
import json

# Notebook Jupyter bosh katalogidan ham, notebooks/ ichidan ham ochilishi mumkin.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "data.yaml").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("configs/data.yaml topilmadi. Notebookni loyiha papkasida oching.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.io_utils import read_jsonl, write_jsonl, write_json
from src.fetch import get_dataset
from src.cleaning import clean_rows
from src.filtering import filter_rows
from src.deduplication import dedup_rows
from src.pii import scrub_rows
from src.split import split_rows
from src.stats import describe_rows, stats_markdown
from src.tokenizer_bpe import SimpleByteBPE
from src.dataset_builder import build_dataset, read_uint16_le

cfg = load_config(ROOT / "configs/data.yaml")
processed = ROOT / cfg["natija"]["processed"]
splits = ROOT / cfg["natija"]["splits"]
reports = ROOT / cfg["natija"]["katalog"]
binary = ROOT / cfg["natija"]["binary"]
print("Loyiha papkasi:", ROOT)
print("Dataset tili:", cfg["loyiha"]["til"])
print("Tokenizer maqsadli hajmi:", cfg["tokenizer"]["vocab_size"])

Loyiha papkasi: c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\6 dars oraliq\uzbek_hikoyalar_midterm\uzbek_hikoyalar_midterm
Dataset tili: uz
Tokenizer maqsadli hajmi: 512


## 0. Repo Structure + Dataset Source Finding / Download

**Repo** — loyiha fayllari jamlangan katalog. **Dataset manbasi** — xom hikoyalar qayerdan olingani. Biz TinyStories yoki internetdagi litsenziyasi noaniq kitobni ko‘chirmadik: hikoyalar `data/source` ichida loyiha bilan birga berilgan.

**Download/Olish bosqichi:** `src/fetch.py` manbani `data/raw` ga ko‘chiradi va **SHA256** (faylni tekshirish uchun raqamli xesh) hisoblaydi. Bu internet talab qilmaydigan, qayta tiklanuvchi yondashuv.

**Tashqi dataset qo‘shishdan oldin**: manba nomi, muallif, litsenziya, foydalanish shartlari va PIIni tekshirish kerak.

In [2]:
# 0-b. Dataset manbasidan xom katalogga nusxa olamiz.
source_file = ROOT / cfg["manba"]["lokal_fayl"]
raw_file = ROOT / cfg["manba"]["raw_fayl"]
source_info = get_dataset(source_file, raw_file)
raw_rows = read_jsonl(raw_file)
print("Yuklangan yozuvlar:", source_info["yozuvlar"])
print("Manba SHA256:", source_info["sha256"][:24] + "...")
print("Birinchi hikoya:", raw_rows[0]["sarlavha"])
print("Matn:", raw_rows[0]["matn"][:220] + "...")

Yuklangan yozuvlar: 94
Manba SHA256: b97d985ecd09851f311fc81f...
Birinchi hikoya: Daraxt ostidagi daftar
Matn: Mahalla kutubxonasi yonidagi chinor tagida Saida eski daftar topdi. Daftarning birinchi betida yo‘qolgan kitoblar ro‘yxati bor edi. Saida ro‘yxatni kutubxonachiga ko‘rsatdi va ular javonlarni birma-bir tekshirishdi. Kech...


## 1. Cleaning — xom matnni tozalash

**Nega kerak?** Turli apostrof belgisi, bo‘sh joy, satrga ko‘chirish va Unicode farqi bir xil matnni har xil ko‘rsatadi.

`src/cleaning.py` quyidagilarni bajaradi: Unicode NFC, `o'`→`o‘`, `g'`→`g‘`, ortiqcha bo‘shliqni bittaga keltirish. Bu bosqich matnning ma'nosini o‘zgartirmaslikka harakat qiladi.

In [3]:
# 1. Har bir xom hikoyani normallashtiramiz.
cleaned = clean_rows(raw_rows)
write_jsonl(processed / "01_cleaned.jsonl", cleaned)
print("Cleaning tugadi:", len(cleaned), "ta yozuv")
print("Fayl: data/processed/01_cleaned.jsonl")

Cleaning tugadi: 94 ta yozuv
Fayl: data/processed/01_cleaned.jsonl


## 2. Filtering — sifatsiz yozuvlarni olib tashlash

**Nega kerak?** Juda qisqa, juda uzun, boshqa til yoki reklamaga o‘xshagan matn modelga foydasiz bo‘lishi mumkin.

`src/filtering.py` `configs/data.yaml` dagi mezonlardan foydalanadi. Har rad etilgan hikoya sabab bilan `results/filtered_out.jsonl` ga yoziladi.

In [4]:
# 2. Faqat mos o‘zbekcha hikoyalarni qoldiramiz.
filtered, rejected, filter_reasons = filter_rows(cleaned, cfg["filtr"])
write_jsonl(processed / "02_filtered.jsonl", filtered)
write_jsonl(reports / "filtered_out.jsonl", rejected)
print("Qoldi:", len(filtered), "| Olib tashlandi:", len(rejected))
print("Sabablar:", filter_reasons)

Qoldi: 92 | Olib tashlandi: 2
Sabablar: {'juda_qisqa': 2}


## 3. Deduplication — bir xil va juda o‘xshash hikoyalarni ajratish

**Nega kerak?** Takroriy matn o‘rganishni bir yoqlama qilishi va train/validation orasiga ma’lumot sizib kirishiga sabab bo‘lishi mumkin.

- **Aniq dublikat:** matn normallashtiriladi va SHA256 xesh bilan solishtiriladi.
- **Yaqin dublikat:** 4 ta ketma-ket so‘zdan iborat *shingles* to‘plami olinadi va **Jaccard** o‘xshashlik koeffitsiyenti hisoblanadi.
- **Katta datasetlar uchun:** MinHash/LSH tezroq bo‘lishi mumkin, lekin bu kichik datasetda oddiy Jaccard tushunarliroq.

In [5]:
# 3. Dublikatlarni ajratib, alohida audit fayliga yozamiz.
unique, duplicates, dup_reasons = dedup_rows(
    filtered,
    n=cfg["dedup"]["shingle_n"],
    threshold=cfg["dedup"]["jaccard_chegara"],
)
write_jsonl(processed / "03_deduplicated.jsonl", unique)
write_jsonl(reports / "duplicates_out.jsonl", duplicates)
print("Noyob hikoyalar:", len(unique))
print("Olib tashlangan dublikatlar:", dup_reasons)

Noyob hikoyalar: 90
Olib tashlangan dublikatlar: {'aniq_dublikat': 1, 'yaqin_dublikat': 1}


## 4. PII Scrubbing — shaxsiy ma’lumotlarni yashirish

**PII** (*Personally Identifiable Information*) — shaxsni aniqlashga yordam beradigan ma'lumotlar. Bu bosqich sinov email va telefon raqamlarini `[EMAIL]`, `[TELEFON]` bilan almashtiradi.

**Cheklov:** regex (muntazam ifoda) hamma ism va manzilni aniqlamaydi. Haqiqiy, maxfiy datasetlarda qo‘shimcha tekshiruv kerak.

In [6]:
# 4. Email va telefon ko‘rinishidagi PII namunalarini berkitamiz.
safe, pii_counts = scrub_rows(unique)
write_jsonl(processed / "04_pii_safe.jsonl", safe)
print("Yashirilgan PII sonlari:", pii_counts)
print("Hikoyalar soni:", len(safe))

Yashirilgan PII sonlari: {'email': 1, 'telefon': 1}
Hikoyalar soni: 90


## 5. Statistics Report + Train/Validation Split

**Statistics Report** — dataset haqida sonli hisobot: hikoyalar, so‘zlar, belgilar, rad etilgan yozuvlar, dublikatlar va PII.

**Train** — model o‘rgatiladigan ma’lumotlar. **Validation** — model natijasini mustaqil tekshirishga ajratilgan ma’lumotlar. Bu ikki bo‘limni hikoya darajasida ajratamiz va doim bir xil chiqishi uchun `seed=42` ishlatamiz.

In [7]:
# 5-a. Hikoyalarni train va validationga ajratamiz.
train, val = split_rows(
    safe,
    val_ratio=cfg["split"]["validation_ulushi"],
    seed=cfg["loyiha"]["seed"],
)
write_jsonl(splits / "train.jsonl", train)
write_jsonl(splits / "val.jsonl", val)
print("Train:", len(train), "| Validation:", len(val))
assert {r["id"] for r in train}.isdisjoint({r["id"] for r in val})

Train: 76 | Validation: 14


In [8]:
# 5-b. Barcha statistikalarni saqlaymiz.
summary = {
    "loyiha": cfg["loyiha"]["nom"],
    "dataset_manzili": str(source_file.relative_to(ROOT)),
    "litsenziya_izohi": "DATASET_README.md: CC0 huquqlar mavjud bo'lgan darajada",
    "manba_sha256": source_info["sha256"],
    "sonlar": {
        "manba": len(raw_rows),
        "cleaning": len(cleaned),
        "filtering": len(filtered),
        "deduplication": len(unique),
        "pii": len(safe),
        "train": len(train),
        "validation": len(val),
    },
    "rad_sabablari": filter_reasons,
    "dublikat_turlari": dup_reasons,
    "pii_topilmalar": pii_counts,
    "umumiy_statistika": describe_rows(safe),
    "train_statistika": describe_rows(train),
    "val_statistika": describe_rows(val),
}
write_json(reports / "stats_report.json", summary)
(reports / "stats_report.md").write_text(stats_markdown(summary), encoding="utf-8")
print("Jami so‘z:", summary["umumiy_statistika"]["jami_soz"])
print("O‘rtacha so‘z:", summary["umumiy_statistika"]["ortalama_soz"])
print("Natijalar: results/stats_report.json va .md")

Jami so‘z: 2764
O‘rtacha so‘z: 30.71
Natijalar: results/stats_report.json va .md


## 6. Tokenizer — matnni token IDlariga aylantirish

**Tokenizer** — matnni model ishlata oladigan butun sonlarga ajratadi. **Byte-Level BPE** (*Byte Pair Encoding*) UTF-8 baytlaridan boshlab, eng ko‘p uchraydigan qo‘shni token juftlarini birlashtirib boradi.

- `vocab_size=512`: maqsadli lug‘at hajmi.
- `min_frequency=2`: juft kamida ikki marta uchragandagina birlashtiriladi.
- `<eos>`: hikoya tugash tokeni, ID = 256.
- **Train-only**: tokenizer validation matnida o‘rgatilmaydi.

**Moslik haqida:** bu aynan o‘rganishga mo‘ljallangan **maxsus JSON format**, Hugging Face `tokenizers` chiqishi emas.

In [9]:
# 6. Byte-Level BPE ni faqat TRAIN hikoyalari asosida o‘qitamiz.
tokenizer = SimpleByteBPE.train(
    [row["matn"] for row in train],
    vocab_size=cfg["tokenizer"]["vocab_size"],
    min_frequency=cfg["tokenizer"]["min_frequency"],
)
tokenizer_path = ROOT / cfg["natija"]["tokenizer_fayl"]
tokenizer.save(tokenizer_path)
print("Tokenizer lug‘ati:", tokenizer.vocab_size)
example = "Bugun bog‘da yangi hikoya boshlandi."
ids = tokenizer.encode(example, add_eos=True)
print("Matn:", example)
print("Token IDlar:", ids)
print("Qayta tiklandi:", tokenizer.decode(ids))
assert tokenizer.decode(ids) == example

Tokenizer lug‘ati: 512
Matn: Bugun bog‘da yangi hikoya boshlandi.
Token IDlar: [66, 117, 103, 294, 361, 301, 275, 458, 311, 104, 288, 309, 258, 493, 306, 264, 256]
Qayta tiklandi: Bugun bog‘da yangi hikoya boshlandi.


## 7. Dataset Builder — train.bin va val.bin

**Dataset Builder** — train/validation hikoyalarini token IDlarga aylantirib, binary faylga yozadi.

- `train.bin`, `val.bin`: **uint16 little-endian** raqamlar ketma-ketligi.
- Har hikoyaning oxiriga `<eos>` qo‘shiladi.
- `meta.json`: tokenlar soni, vocab hajmi, format, SHA256.

**Nega kerak?** Keyingi GPT Training Loop tokenlarni tayyor binary fayldan tez o‘qishi mumkin.

In [10]:
# 7. Token IDlarni saqlaymiz va qayta o‘qib tekshiramiz.
built = build_dataset(train, val, tokenizer, binary)
train_ids = read_uint16_le(binary / "train.bin")
val_ids = read_uint16_le(binary / "val.bin")
print("Training tokenlari:", len(train_ids))
print("Validation tokenlari:", len(val_ids))
print("Binary turi:", built["format"])
print("EOS token soni (train):", train_ids.count(256))
assert train_ids.count(256) == len(train)
assert val_ids.count(256) == len(val)

Training tokenlari: 7719
Validation tokenlari: 1525
Binary turi: uint16_le
EOS token soni (train): 76


## 8. Yakuniy Readiness — natijalarni tekshirish

Quyidagi tekshiruvlar `train.bin`, `val.bin`, `tokenizer.json`, statistik hisobot va train/validation ajratish to‘g‘riligini ko‘radi. **Bu modelni o‘qitish emas; datasetning tayyorligini tekshirishdir.**

In [11]:
# 8. Midterm natijalarini bir martada tekshirish.
expected = [
    raw_file,
    processed / "01_cleaned.jsonl",
    processed / "02_filtered.jsonl",
    processed / "03_deduplicated.jsonl",
    processed / "04_pii_safe.jsonl",
    splits / "train.jsonl",
    splits / "val.jsonl",
    tokenizer_path,
    binary / "train.bin",
    binary / "val.bin",
    binary / "meta.json",
    reports / "stats_report.json",
    reports / "stats_report.md",
    ROOT / "DATASET_README.md",
]
for path in expected:
    assert path.is_file(), f"Topilmadi: {path}"
assert len(train) > 0 and len(val) > 0
assert tokenizer.decode(tokenizer.encode("Salom, o‘zbekcha hikoya!")) == "Salom, o‘zbekcha hikoya!"
assert max(train_ids + val_ids) < tokenizer.vocab_size
assert "test@example.com" not in " ".join(r["matn"] for r in safe)
print(f"✅ HAMMASI TAYYOR: {len(expected)} ta majburiy fayl tekshirildi.")
print("✅ Dataset Buildergacha barcha bosqichlar bajarildi.")
print("💡 Datasetning litsenziyasi: DATASET_README.md")

✅ HAMMASI TAYYOR: 14 ta majburiy fayl tekshirildi.
✅ Dataset Buildergacha barcha bosqichlar bajarildi.
💡 Datasetning litsenziyasi: DATASET_README.md


## Qisqa xulosa va keyingi bosqich

Tayyor fayllar: `data/binaries/train.bin`, `data/binaries/val.bin`, `data/binaries/meta.json`, `tokenizer/tokenizer.json` va `results/stats_report.md`.

**Keyingi dars:** `model_config.py`, `model.py` va Training Loop yordamida GPTni o‘qitish. **Bu notebook ataylab Dataset Builderda yakunlanadi** — Midterm talabi shu chegaragacha.

**Mavjud bo‘lmagan imkoniyatlar:** real web-scraping, millionlab hikoya, yirik LLM treningi, professional PII detektori va Hugging Face tokenizer formati. Bular bosqichma-bosqich keyin qo‘shiladi.